# Hands-on Pertemuan 1 · Data dalam Bisnis Digital
**Business Statistics and Data Visualization · S1 Bisnis Digital · Sub-CPMK-1**

Notebook ini adalah versi Python (Google Colab) dari praktikum lab *"Eksplorasi dataset e-commerce"*. Langkahnya sama dengan yang di spreadsheet: membuka data, mengenali tipe kolom, memfilter, meringkas dengan pivot table, lalu membuat grafik pertama. Ditambah latihan kualitas data dan pseudonimisasi sesuai UU PDP.

| Bagian | Isi | Kaitan dengan slide |
|---|---|---|
| 1 | Persiapan | – |
| 2 | Membuka dan mengenali dataset | Praktikum langkah 1–2 |
| 3 | Jenis data dan skala pengukuran (NOIR) | Slide 12–18 |
| 4 | Filter dan sort | Praktikum langkah 4 |
| 5 | Pivot table dan grafik batang | Praktikum langkah 5–6 |
| 6 | Struktur data: cross-section, time series, panel | Slide 15 |
| 7 | Statistika deskriptif vs inferensial | Slide 8 |
| 8 | Menjawab pertanyaan bisnis | Slide 7, 9 |
| 9 | Memeriksa kualitas data | Slide 22 |
| 10 | Etika data dan pseudonimisasi | Slide 24–26 |
| 11 | Latihan mandiri | – |
| 12 | Persiapan Tugas 1 | Slide 29 |

**Cara pakai:** jalankan sel dari atas ke bawah (`Shift + Enter`). Sel bertanda ✏️ **LATIHAN** perlu kamu isi sendiri. Semua data di notebook ini **fiktif** dan dibuat dengan kode, jadi hasilnya sama untuk semua mahasiswa.

## 1. Persiapan
Kita memakai tiga library utama:
- **pandas**: mengolah tabel (setara spreadsheet di Python)
- **numpy**: perhitungan angka
- **matplotlib**: membuat grafik

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def rp(x):
    '''Format angka menjadi Rupiah, contoh: 1250000 -> Rp1.250.000'''
    return "Rp" + f"{x:,.0f}".replace(",", ".")

print("Library siap. pandas versi", pd.__version__)

## 2. Membuka dan mengenali dataset

### 2.1 Membuat dataset e-commerce
Sel berikut membuat **1.000 pesanan fiktif** dari sebuah toko online selama September 2026, dengan kolom yang sama seperti latihan di slide 17, ditambah beberapa kolom lain.

> Tidak perlu memahami semua baris kode di sel ini. Anggap saja ini "mengunduh dataset dari LMS". Fokus kita mulai dari bagian 2.2.

In [ ]:
rng = np.random.default_rng(2026)   # seed tetap agar hasil semua mahasiswa sama
n = 1000

kategori_list = ["Fashion", "Elektronik", "Kecantikan", "Rumah tangga"]
kategori = rng.choice(kategori_list, size=n, p=[0.41, 0.19, 0.26, 0.14])

kota_prefix = {"Makassar": 90, "Jakarta": 10, "Surabaya": 60,
               "Medan": 20, "Bandung": 40, "Denpasar": 80}
kota = rng.choice(list(kota_prefix), size=n, p=[0.22, 0.28, 0.17, 0.12, 0.13, 0.08])
kode_pos = [f"{kota_prefix[k]}{rng.integers(100, 999)}" for k in kota]

# Harga: tiap kategori punya kisaran berbeda (dibulatkan ke ratusan rupiah)
median_harga = {"Fashion": 160_000, "Elektronik": 900_000,
                "Kecantikan": 85_000, "Rumah tangga": 70_000}
harga = np.array([rng.lognormal(np.log(median_harga[k]), 0.45) for k in kategori])
harga = (np.round(harga / 500) * 500).astype(int)

jumlah = rng.poisson(0.9, n) + 1                              # minimal 1 barang
ukuran = np.where(kategori == "Fashion",
                  rng.choice(["S", "M", "L", "XL"], size=n, p=[0.2, 0.4, 0.3, 0.1]),
                  None)
metode_bayar = rng.choice(["E-wallet", "Transfer bank", "COD", "Kartu kredit", "Paylater"],
                          size=n, p=[0.38, 0.22, 0.2, 0.08, 0.12])

# Lama kirim lebih panjang untuk kota di luar Jawa
jauh = np.isin(kota, ["Makassar", "Medan", "Denpasar"])
lama_kirim = rng.poisson(np.where(jauh, 4, 2)) + 1

# Rating cenderung turun bila pengiriman lama
skor = 4.6 - 0.18 * lama_kirim + rng.normal(0, 0.8, n)
rating = np.clip(np.round(skor), 1, 5).astype(int)

tanggal = pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 30, n), unit="D")
urut = np.argsort(tanggal.values, kind="stable")

df = pd.DataFrame({
    "order_id": [f"ORD-{10231 + i}" for i in range(n)],
    "tanggal_pesan": tanggal.values[urut],
    "kategori": kategori[urut],
    "ukuran": ukuran[urut],
    "kota": kota[urut],
    "kode_pos": np.array(kode_pos)[urut],
    "metode_bayar": metode_bayar[urut],
    "harga": harga[urut],
    "jumlah": jumlah[urut],
    "lama_kirim_hari": lama_kirim[urut],
    "rating": rating[urut],
})

# Simpan sebagai file, supaya bisa juga dibuka di Excel / Google Sheets
df.to_excel("dataset_ecommerce.xlsx", index=False)
df.to_csv("dataset_ecommerce.csv", index=False)
print("Dataset tersimpan: dataset_ecommerce.xlsx dan dataset_ecommerce.csv")

### 2.2 Membaca file ke dalam DataFrame
Di spreadsheet kita membuka file dengan *File → Open*. Di Python kita memakai `pd.read_excel()` atau `pd.read_csv()`.

> 💡 Jika dosen membagikan `dataset_ecommerce.xlsx` di LMS, unggah file itu lewat ikon folder di panel kiri Colab, lalu jalankan sel di bawah. Nama kolom di file LMS bisa sedikit berbeda, jadi sesuaikan kode berikutnya bila perlu.

In [ ]:
df = pd.read_excel("dataset_ecommerce.xlsx", dtype={"kode_pos": str})
df.head()      # 5 baris pertama, seperti melihat bagian atas spreadsheet

### 2.3 Berapa baris dan kolom?
Di spreadsheet, kita menggulir ke bawah untuk melihat baris terakhir. Di pandas cukup `.shape`.

In [ ]:
baris, kolom = df.shape
print(f"Dataset berisi {baris} baris (pesanan) dan {kolom} kolom")
print("Nama kolom:", list(df.columns))

In [ ]:
df.tail(3)     # 3 baris terakhir

### 2.4 Tipe kolom menurut komputer
`.info()` menampilkan tipe setiap kolom dan jumlah nilai yang terisi (non-null).
- `int64` / `float64` = angka
- `object` atau `str` = teks
- `datetime64` = tanggal

**Perhatikan:** tipe menurut komputer **belum tentu** sama dengan jenis data secara statistik. Itulah yang kita periksa di bagian 3.

In [ ]:
df.info()

**Pertanyaan refleksi:** kolom `ukuran` hanya terisi sebagian (sekitar 38%). Apakah itu masalah kualitas data? *(Petunjuk: ukuran hanya berlaku untuk produk Fashion.)*

## 3. Jenis data dan skala pengukuran (NOIR)

Ingat pohon jenis data (slide 12) dan skala NOIR (slide 13):

| Skala | Sifat | Contoh | Statistik yang tepat |
|---|---|---|---|
| **N**ominal | Membedakan kategori | metode bayar, kota | modus, frekuensi |
| **O**rdinal | + ada urutan | rating ★1–5, ukuran S/M/L | + median, persentil |
| **I**nterval | + jarak antarnilai sama | tanggal, suhu °C | + mean, simpangan baku |
| **R**asio | + punya nol mutlak | harga, jumlah | + rasio ("2 kali lipat") |

### 3.1 ✏️ LATIHAN CEPAT (5 menit)
Isi dictionary berikut untuk setiap kolom. Pilihan jenis: `"kategorik"`, `"numerik diskrit"`, `"numerik kontinu"`. Pilihan skala: `"nominal"`, `"ordinal"`, `"interval"`, `"rasio"`.

> Petunjuk: tidak semua angka adalah data numerik.

In [ ]:
jawaban_saya = {
    # "kolom":          ("jenis",       "skala"),
    "order_id":         ("",            ""),
    "tanggal_pesan":    ("",            ""),
    "kategori":         ("",            ""),
    "ukuran":           ("",            ""),
    "kota":             ("",            ""),
    "kode_pos":         ("",            ""),
    "metode_bayar":     ("",            ""),
    "harga":            ("",            ""),
    "jumlah":           ("",            ""),
    "lama_kirim_hari":  ("",            ""),
    "rating":           ("",            ""),
}

Jalankan sel berikut **setelah** kamu mengisi jawaban. Sel ini membandingkan jawabanmu dengan kunci dari slide 18.

In [ ]:
kunci = {
    "order_id":        ("kategorik",       "nominal",  "Label unik; tidak untuk dihitung"),
    "tanggal_pesan":   ("numerik",         "interval", "Selisih bermakna, tanpa nol mutlak"),
    "kategori":        ("kategorik",       "nominal",  "Kelompok tanpa urutan"),
    "ukuran":          ("kategorik",       "ordinal",  "S < M < L < XL, jarak tidak pasti"),
    "kota":            ("kategorik",       "nominal",  "Kelompok tanpa urutan"),
    "kode_pos":        ("kategorik",       "nominal",  "Angka sebagai label lokasi"),
    "metode_bayar":    ("kategorik",       "nominal",  "Kelompok tanpa urutan"),
    "harga":           ("numerik kontinu", "rasio",    "Nol mutlak; '2x lebih mahal' bermakna"),
    "jumlah":          ("numerik diskrit", "rasio",    "Hasil menghitung; nol = tidak membeli"),
    "lama_kirim_hari": ("numerik diskrit", "rasio",    "Dicatat dalam hari bulat; nol = sampai hari itu juga"),
    "rating":          ("kategorik",       "ordinal",  "Berurutan; sering didekati sebagai interval"),
}

hasil = []
for kol, (jenis_k, skala_k, alasan) in kunci.items():
    jenis_s, skala_s = jawaban_saya.get(kol, ("", ""))
    j, s = jenis_s.strip().lower(), skala_s.strip().lower()
    jenis_benar = (j == jenis_k) or (jenis_k == "numerik" and j.startswith("numerik"))
    benar = jenis_benar and s == skala_k
    hasil.append({"kolom": kol, "jawaban_kamu": f"{jenis_s} / {skala_s}",
                  "kunci": f"{jenis_k} / {skala_k}", "alasan": alasan,
                  "cek": "✅" if benar else "❌"})

hasil = pd.DataFrame(hasil)
print(f"Skor: {(hasil['cek'] == '✅').sum()} dari {len(hasil)} kolom")
hasil

### 3.2 Memberi tahu pandas jenis data yang benar
Supaya analisis tidak keliru, kita ubah tipe kolom sesuai jenis datanya:
- `kode_pos` → teks (bukan angka)
- `ukuran` dan `rating` → kategori **berurutan** (ordinal)
- kolom nominal lain → kategori

In [ ]:
df["kode_pos"] = df["kode_pos"].astype(str)
df["ukuran"] = pd.Categorical(df["ukuran"], categories=["S", "M", "L", "XL"], ordered=True)
df["rating"] = pd.Categorical(df["rating"], categories=[1, 2, 3, 4, 5], ordered=True)
for kol in ["kategori", "kota", "metode_bayar"]:
    df[kol] = df[kol].astype("category")

df.dtypes

### 3.3 Skala menentukan analisis: contoh yang **keliru**
Komputer akan tetap menghitung apa pun yang kita minta, meskipun hasilnya tidak bermakna.

In [ ]:
# KELIRU 1: rata-rata kode pos
kode_pos_angka = pd.to_numeric(df["kode_pos"])
print("Rata-rata kode pos =", round(kode_pos_angka.mean()), "  <- tidak bermakna, ini hanya label lokasi")

# KELIRU 2: menganggap rating sebagai rasio
print("Rating 4 / rating 2 =", 4 / 2, "  <- TIDAK berarti '2 kali lebih puas'")

### 3.4 Ringkasan yang **tepat** untuk tiap skala

In [ ]:
# NOMINAL -> frekuensi dan modus
print("=== NOMINAL: metode_bayar ===")
print(df["metode_bayar"].value_counts())
print("Modus:", df["metode_bayar"].mode()[0])

In [ ]:
# ORDINAL -> frekuensi berurutan, median, persentase
print("=== ORDINAL: rating ===")
print(df["rating"].value_counts().sort_index())

rating_angka = df["rating"].astype(int)
print("\nMedian rating:", rating_angka.median())
print(f"Ulasan bernilai ★4 ke atas: {(rating_angka >= 4).mean():.0%}")

In [ ]:
# INTERVAL -> selisih bermakna (tapi tidak ada 'dua kali lipat tanggal')
print("=== INTERVAL: tanggal_pesan ===")
print("Pesanan pertama :", df["tanggal_pesan"].min().date())
print("Pesanan terakhir:", df["tanggal_pesan"].max().date())
print("Rentang         :", (df["tanggal_pesan"].max() - df["tanggal_pesan"].min()).days, "hari")

In [ ]:
# RASIO -> mean, simpangan baku, dan perbandingan rasio bermakna
print("=== RASIO: harga ===")
print("Rata-rata harga :", rp(df["harga"].mean()))
print("Median harga    :", rp(df["harga"].median()))
print("Simpangan baku  :", rp(df["harga"].std()))

h = df.groupby("kategori", observed=True)["harga"].median()
print(f"\nMedian harga Elektronik {h['Elektronik'] / h['Fashion']:.1f}x median harga Fashion  <- bermakna di skala rasio")

### 3.5 Grafik juga mengikuti jenis data
- Kategorik (nominal/ordinal) → **diagram batang** (ordinal: urutan batang mengikuti urutan kategori)
- Numerik kontinu → **histogram**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["kota"].value_counts().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Nominal: pesanan per kota\n(urut dari terbanyak)")
axes[0].set_ylabel("Jumlah pesanan")

df["rating"].value_counts().sort_index().plot.bar(ax=axes[1], color="#DD8452")
axes[1].set_title("Ordinal: distribusi rating\n(urut 1 → 5, bukan dari terbanyak)")

axes[2].hist(df["harga"] / 1000, bins=40, color="#55A868")
axes[2].set_title("Rasio: distribusi harga")
axes[2].set_xlabel("Harga (ribu Rp)")

for ax in axes[:2]:
    ax.tick_params(axis="x", rotation=0)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 4. Filter dan sort
Di spreadsheet: *Data → Filter* dan *Sort*. Di pandas: `sort_values()` dan penyaringan dengan kondisi.

### 4.1 Sort: 10 produk termahal

In [ ]:
df.sort_values("harga", ascending=False).head(10)[["order_id", "kategori", "kota", "harga"]]

### 4.2 Kota dengan pesanan terbanyak

In [ ]:
df["kota"].value_counts()

### 4.3 Filter dengan satu dan beberapa kondisi
- `&` berarti **dan**, `|` berarti **atau**
- setiap kondisi dibungkus tanda kurung

In [ ]:
# Pesanan Fashion dari Makassar
fashion_mks = df[(df["kategori"] == "Fashion") & (df["kota"] == "Makassar")]
print("Pesanan Fashion dari Makassar:", len(fashion_mks))
fashion_mks.head()

In [ ]:
# Ulasan buruk (rating 1 atau 2): dari mana saja?
rating_buruk = df[df["rating"].astype(int) <= 2]
print("Jumlah ulasan buruk:", len(rating_buruk))
rating_buruk["kota"].value_counts()

### ✏️ LATIHAN 4
1. Tampilkan 5 pesanan **Elektronik** termurah.
2. Berapa pesanan yang dibayar dengan **COD** dan harganya di atas Rp500.000?

In [ ]:
# Jawaban 1

In [ ]:
# Jawaban 2

## 5. Pivot table dan grafik batang

### 5.1 Membuat kolom omzet
Omzet per pesanan = harga × jumlah. Sama seperti menambah kolom rumus di spreadsheet.

In [ ]:
df["omzet"] = df["harga"] * df["jumlah"]
df[["order_id", "harga", "jumlah", "omzet"]].head()

### 5.2 Pivot: pesanan dan omzet per kategori
Setara dengan *Insert → Pivot table* dengan **Rows = kategori** dan **Values = COUNT(order_id), SUM(omzet)**.

In [ ]:
pivot_kategori = pd.pivot_table(
    df,
    index="kategori",
    values=["order_id", "omzet"],
    aggfunc={"order_id": "count", "omzet": "sum"},
    observed=True,
).rename(columns={"order_id": "pesanan"}).sort_values("omzet", ascending=False)

pivot_kategori["porsi_omzet"] = pivot_kategori["omzet"] / pivot_kategori["omzet"].sum()
pivot_kategori["rata2_omzet_per_pesanan"] = pivot_kategori["omzet"] / pivot_kategori["pesanan"]

pivot_kategori.style.format({
    "omzet": rp, "rata2_omzet_per_pesanan": rp, "porsi_omzet": "{:.1%}"
})

**Pertanyaan refleksi:** kategori mana yang pesanannya paling banyak? Apakah kategori itu juga penyumbang omzet terbesar? Mengapa bisa berbeda?

### 5.3 Pivot dua dimensi: kategori × kota
Setara dengan **Rows = kategori**, **Columns = kota**.

In [ ]:
pivot_kk = pd.pivot_table(df, index="kategori", columns="kota", values="order_id",
                          aggfunc="count", margins=True, margins_name="Total", observed=True)
pivot_kk

### 5.4 Grafik batang pertama dari pivot table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pesanan = pivot_kategori["pesanan"].sort_values()
axes[0].barh(pesanan.index, pesanan.values, color="#4C72B0")
axes[0].set_title("Jumlah pesanan per kategori")
axes[0].set_xlabel("Pesanan")
for i, v in enumerate(pesanan.values):
    axes[0].text(v + 5, i, str(v), va="center")

omzet_juta = (pivot_kategori["omzet"] / 1e6).sort_values()
axes[1].barh(omzet_juta.index, omzet_juta.values, color="#DD8452")
axes[1].set_title("Omzet per kategori")
axes[1].set_xlabel("Omzet (juta Rp)")
for i, v in enumerate(omzet_juta.values):
    axes[1].text(v + 2, i, f"{v:,.0f}", va="center")

plt.suptitle("Toko online fiktif · September 2026", y=1.02)
plt.tight_layout()
plt.show()

### ✏️ LATIHAN 5
Buat pivot table **omzet per metode bayar** (urut dari terbesar), lalu buat grafik batangnya.

In [ ]:
# Jawaban

## 6. Struktur data menurut waktu (slide 15)

| Struktur | Ciri | Contoh dari dataset ini |
|---|---|---|
| Cross-section | Banyak objek, satu waktu | Omzet per kota selama September |
| Time series | Satu objek, banyak waktu | Omzet harian seluruh toko |
| Panel | Banyak objek, banyak waktu | Omzet mingguan per kota |

In [ ]:
# Cross-section
cross = df.groupby("kota", observed=True)["omzet"].sum().sort_values(ascending=False) / 1e6

# Time series
ts = df.groupby("tanggal_pesan")["omzet"].sum() / 1e6

# Panel: omzet mingguan per kota
df["minggu"] = df["tanggal_pesan"].dt.to_period("W").dt.start_time
panel = df.pivot_table(index="minggu", columns="kota", values="omzet",
                       aggfunc="sum", observed=True) / 1e6

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cross.plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Cross-section: omzet per kota")
axes[0].tick_params(axis="x", rotation=30)

ts.plot(ax=axes[1], marker="o", color="#DD8452")
axes[1].set_title("Time series: omzet harian")

panel.plot(ax=axes[2], marker="o")
axes[2].set_title("Panel: omzet mingguan per kota\n(minggu terakhir hanya 3 hari)")
axes[2].legend(fontsize=8, ncol=2)

for ax in axes:
    ax.set_ylabel("Juta Rp")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

print("Bentuk data panel (baris = minggu, kolom = kota):")
panel.round(1)

## 7. Statistika deskriptif vs inferensial (slide 8)

- **Deskriptif**: meringkas data yang kita miliki.
- **Inferensial**: menebak kondisi **populasi** dari **sampel**, lengkap dengan tingkat keyakinan.

Simulasi: anggap 1.000 pesanan ini adalah **seluruh populasi** pelanggan. Kita hanya "mensurvei" 100 pelanggan secara acak, lalu melihat seberapa dekat hasil sampel dengan nilai populasi. *(Rumus selang kepercayaan dibahas lengkap di minggu 6.)*

In [ ]:
populasi = df["rating"].astype(int)
print(f"DESKRIPTIF (seluruh populasi) -> rata-rata rating = {populasi.mean():.2f}")

sampel = populasi.sample(100, random_state=7)
rata2 = sampel.mean()
se = sampel.std() / np.sqrt(len(sampel))
bawah, atas = rata2 - 1.96 * se, rata2 + 1.96 * se
print(f"INFERENSIAL (sampel 100)      -> rata-rata sampel = {rata2:.2f}")
print(f"                                 95% yakin rata-rata populasi antara {bawah:.2f} dan {atas:.2f}")

In [ ]:
# Ulangi survei 50 kali: seberapa sering selang kepercayaan memuat nilai populasi?
mu = populasi.mean()
kena = 0
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(50):
    s = populasi.sample(100, random_state=100 + i)
    m, e = s.mean(), 1.96 * s.std() / np.sqrt(100)
    memuat = (m - e) <= mu <= (m + e)
    kena += memuat
    ax.plot([m - e, m + e], [i, i], color="#4C72B0" if memuat else "#C44E52", lw=2)
    ax.plot(m, i, "o", color="black", ms=3)
ax.axvline(mu, color="black", ls="--", label=f"Rata-rata populasi = {mu:.2f}")
ax.set_xlabel("Rata-rata rating")
ax.set_ylabel("Survei ke-")
ax.set_title(f"{kena} dari 50 selang kepercayaan 95% memuat nilai populasi")
ax.legend()
plt.show()

## 8. Dari data ke keputusan (slide 7 dan 9)

Pertanyaan bisnis dari slide 7: **"Apakah pengiriman lebih cepat menaikkan rating?"**

Kita jawab di tingkat **deskriptif** (apa yang terjadi) dan **diagnostik** (mengapa terjadi).

In [ ]:
ringkas = df.assign(rating_angka=df["rating"].astype(int)).groupby("lama_kirim_hari").agg(
    pesanan=("order_id", "count"),
    rata2_rating=("rating_angka", "mean"),
    persen_rating_4_ke_atas=("rating_angka", lambda r: (r >= 4).mean()),
)
ringkas = ringkas[ringkas["pesanan"] >= 20]   # buang kelompok yang terlalu kecil
ringkas.style.format({"rata2_rating": "{:.2f}", "persen_rating_4_ke_atas": "{:.0%}"})

In [ ]:
fig, ax = plt.subplots()
ax.bar(ringkas.index.astype(str), ringkas["persen_rating_4_ke_atas"], color="#55A868")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("Lama kirim (hari)")
ax.set_ylabel("Ulasan ★4 ke atas")
ax.set_title("Semakin lama pengiriman, semakin sedikit ulasan puas")
plt.show()

In [ ]:
# Diagnostik: kota mana yang pengirimannya paling lama dan ratingnya paling rendah?
df.assign(rating_angka=df["rating"].astype(int)).groupby("kota", observed=True).agg(
    rata2_lama_kirim=("lama_kirim_hari", "mean"),
    rata2_rating=("rating_angka", "mean"),
).sort_values("rata2_lama_kirim", ascending=False).round(2)

**Diskusi:**
1. Kota mana yang sebaiknya diprioritaskan untuk perbaikan logistik?
2. Apakah grafik di atas **membuktikan** bahwa pengiriman lama *menyebabkan* rating turun? Faktor lain apa yang mungkin berperan? *(Hubungan sebab-akibat dan regresi dibahas di minggu 10–11.)*

## 9. Memeriksa kualitas data (slide 22)

*Garbage in, garbage out.* Tabel berikut sama dengan contoh di slide. Tugas kita di pertemuan ini baru **menemukan** masalah; cara **membersihkannya** dipraktikkan di minggu 2.

In [ ]:
kotor = pd.DataFrame({
    "order_id": ["ORD-2001", "ORD-2002", "ORD-2002", "ORD-2003", "ORD-2004"],
    "tanggal":  ["2026-09-01", "01/09/2026", "01/09/2026", "2026-09-02", "2026-09-02"],
    "kota":     ["Makassar", "mks", "mks", None, "Jakarta"],
    "harga":    [149000, 89500, 89500, "Rp 312.000", 215000],
    "rating":   [5, 4, 4, 3, 7],
})
kotor

In [ ]:
masalah = []

# 1. Baris duplikat (dimensi: akurat)
dup = kotor[kotor.duplicated(keep=False)]
masalah.append(("Akurat", "Baris duplikat", list(dup.index)))

# 2. Nilai kosong (dimensi: lengkap)
kosong = kotor[kotor.isna().any(axis=1)]
masalah.append(("Lengkap", "Nilai kosong", list(kosong.index)))

# 3. Format tanggal tidak seragam (dimensi: konsisten)
tgl_iso = pd.to_datetime(kotor["tanggal"], format="%Y-%m-%d", errors="coerce")
masalah.append(("Konsisten", "Format tanggal tidak seragam", list(kotor.index[tgl_iso.isna()])))

# 4. Nama kota tidak baku (dimensi: konsisten)
kota_baku = {"Makassar", "Jakarta", "Surabaya", "Medan", "Bandung", "Denpasar"}
tidak_baku = kotor["kota"].notna() & ~kotor["kota"].isin(kota_baku)
masalah.append(("Konsisten", "Nama kota tidak baku", list(kotor.index[tidak_baku])))

# 5. Harga tersimpan sebagai teks (dimensi: valid)
harga_num = pd.to_numeric(kotor["harga"], errors="coerce")
masalah.append(("Valid", "Harga bukan angka", list(kotor.index[harga_num.isna()])))

# 6. Rating di luar rentang 1-5 (dimensi: valid)
luar = ~kotor["rating"].between(1, 5)
masalah.append(("Valid", "Rating di luar rentang 1–5", list(kotor.index[luar])))

laporan = pd.DataFrame(masalah, columns=["dimensi", "masalah", "indeks_baris"])
laporan

**Dimensi kelima, *tepat waktu*,** tidak bisa dicek dari isi tabel saja. Kita perlu tahu kapan data diambil dan untuk keputusan apa data itu dipakai. Contoh: data harga bulan lalu mungkin sudah tidak relevan untuk mengatur promo Harbolnas.

### ✏️ LATIHAN 9
Jalankan pemeriksaan yang sama pada dataset utama `df`. Apakah ada duplikat? Nilai kosong? Rating di luar rentang? Apakah nilai kosong di kolom `ukuran` termasuk masalah kualitas data?

In [ ]:
# Jawaban

## 10. Etika data dan pseudonimisasi (slide 24–26)

Tim marketing ingin data pelanggan untuk analisis penjualan per kota. Data mentahnya memuat **data pribadi**. Sesuai prinsip **minimisasi** dan **pseudonimisasi**, kita samarkan dulu sebelum dibagikan ke tim analis.

> Semua nama, nomor HP, dan alamat di bawah ini **fiktif**.

In [ ]:
pelanggan = pd.DataFrame({
    "nama":        ["Siti Rahma", "Budi Santoso", "Ayu Lestari", "Andi Pratama", "Made Wirawan"],
    "no_hp":       ["0812-3456-7810", "0857-1122-3344", "0813-9988-7766", "0821-5544-3322", "0878-6655-4411"],
    "alamat":      ["Jl. Melati 12, Makassar", "Jl. Kenanga 5, Medan", "Jl. Mawar 8, Bandung",
                    "Jl. Anggrek 21, Makassar", "Jl. Sudirman 3, Denpasar"],
    "tgl_lahir":   ["1998-04-12", "1985-11-02", "2001-07-30", "1993-01-15", "1979-09-09"],
    "riwayat_kesehatan": ["-", "Diabetes", "-", "-", "Hipertensi"],
    "belanja":     [1_250_000, 480_000, 735_000, 2_100_000, 390_000],
})
pelanggan

### 10.1 Identifikasi data pribadi (UU No. 27 Tahun 2022, Pasal 4)

In [ ]:
klasifikasi_pdp = pd.DataFrame([
    ("nama",              "Data pribadi umum",     "Pasal 4 ayat (3)"),
    ("no_hp",             "Data pribadi umum*",    "Dapat mengidentifikasi seseorang bila dikombinasikan"),
    ("alamat",            "Data pribadi umum*",    "Dapat mengidentifikasi seseorang bila dikombinasikan"),
    ("tgl_lahir",         "Data pribadi umum*",    "Dapat mengidentifikasi seseorang bila dikombinasikan"),
    ("riwayat_kesehatan", "Data pribadi SPESIFIK", "Pasal 4 ayat (2): risiko lebih tinggi"),
    ("belanja",           "Bukan identitas langsung", "Tetap sensitif bila tertaut ke identitas"),
], columns=["kolom", "status", "keterangan"])
klasifikasi_pdp

### 10.2 Minimisasi: ambil hanya kolom yang dibutuhkan
Untuk analisis penjualan per kota, tim **tidak** butuh riwayat kesehatan dan tanggal lahir. Jangan ikut dibagikan.

### 10.3 Pseudonimisasi
- **Nama** → diganti kode pelanggan (hash dengan *salt* rahasia, sehingga tidak bisa ditebak balik tanpa kunci)
- **No. HP** → disamarkan bagian tengahnya
- **Alamat** → diringkas menjadi kota saja

In [ ]:
import hashlib

SALT = "ganti-dengan-kunci-rahasia"   # disimpan terpisah, tidak ikut dibagikan

def kode_pelanggan(nama):
    h = hashlib.sha256((SALT + nama).encode()).hexdigest()
    return "CUST-" + h[:4].upper()

def samarkan_hp(no_hp):
    bagian = no_hp.split("-")
    return f"{bagian[0]}-****-{bagian[-1]}"

def ambil_kota(alamat):
    return alamat.split(",")[-1].strip()

aman = pd.DataFrame({
    "id_pelanggan": pelanggan["nama"].apply(kode_pelanggan),
    "no_hp":        pelanggan["no_hp"].apply(samarkan_hp),
    "kota":         pelanggan["alamat"].apply(ambil_kota),
    "belanja":      pelanggan["belanja"],
})
aman

In [ ]:
# Analisis tetap bisa dilakukan tanpa mengetahui identitas pelanggan
aman.groupby("kota")["belanja"].agg(["count", "sum"]).rename(
    columns={"count": "pelanggan", "sum": "total_belanja"})

**Catatan penting:**
- Data hasil pseudonimisasi **masih** tergolong data pribadi, karena bisa dikembalikan dengan kunci (`SALT`). Kunci wajib disimpan aman dan terpisah.
- Prinsip **keamanan**: jangan kirim file mentah lewat grup chat, batasi siapa yang boleh mengakses.
- Bila terjadi kebocoran, UU PDP Pasal 46 mewajibkan pemberitahuan tertulis paling lambat **3×24 jam**.

### ✏️ DISKUSI: kasus "TokoKita" (slide 27)
Tulis jawaban kelompokmu di sel markdown di bawah ini (klik dua kali untuk mengedit).

1. **Data apa yang bocor?** Mana yang umum, mana yang spesifik?
   - ...
2. **Kewajiban apa yang dilanggar?**
   - ...
3. **Siapa yang dirugikan?**
   - ...
4. **Tiga langkah pencegahan dan tiga langkah respons:**
   - Pencegahan: ...
   - Respons: ...

## 11. ✏️ Latihan mandiri
Gunakan dataset `df`.

1. Kategori apa yang **paling sering** dibeli di Denpasar? *(nominal → modus)*
2. Berapa **median** ukuran yang dibeli untuk produk Fashion? *(ordinal → median)* Petunjuk: `df["ukuran"].cat.codes` memberi nomor urut; atau gunakan `.value_counts().sort_index().cumsum()`.
3. Berapa **rata-rata** jumlah barang per pesanan untuk tiap metode bayar? *(rasio → mean)*
4. Buat pivot **rata-rata rating per kategori × kota**. Kombinasi mana yang paling rendah?
5. Buat satu grafik yang menjawab: *"Hari apa dalam seminggu yang omzetnya paling besar?"* Petunjuk: `df["tanggal_pesan"].dt.day_name()`.

In [ ]:
# Jawaban 1

In [ ]:
# Jawaban 2

In [ ]:
# Jawaban 3

In [ ]:
# Jawaban 4

In [ ]:
# Jawaban 5

## 12. Persiapan Tugas 1: klasifikasi dataset Kaggle

Tugas 1 (individu, dikumpulkan sebelum pertemuan 2): pilih dataset e-commerce dari Kaggle (**minimal 8 kolom, 500 baris**), lalu buat tabel klasifikasi, identifikasi masalah kualitas data, tandai data pribadi, dan tulis dua pertanyaan bisnis.

Fungsi di bawah membantu membuat **draf** profil kolom. Hasilnya hanya titik awal: **jenis dan skala data tetap harus kamu tentukan sendiri** dengan alasan, karena komputer tidak tahu makna kolom (ingat kasus `kode_pos`).

In [ ]:
def profil_kolom(data):
    '''Draf profil setiap kolom: tipe komputer, nilai unik, nilai kosong, contoh isi.'''
    baris = []
    for kol in data.columns:
        s = data[kol]
        baris.append({
            "kolom": kol,
            "tipe_komputer": str(s.dtype),
            "nilai_unik": s.nunique(),
            "kosong": int(s.isna().sum()),
            "persen_kosong": f"{s.isna().mean():.1%}",
            "contoh": ", ".join(map(str, s.dropna().unique()[:3])),
            "jenis_data (isi sendiri)": "",
            "skala (isi sendiri)": "",
            "alasan (isi sendiri)": "",
        })
    return pd.DataFrame(baris)

def cek_kualitas(data):
    '''Pemeriksaan kualitas data dasar.'''
    print(f"Ukuran data     : {data.shape[0]} baris x {data.shape[1]} kolom")
    print(f"Baris duplikat  : {data.duplicated().sum()}")
    print(f"Sel kosong      : {int(data.isna().sum().sum())}")
    teks_angka = [k for k in data.select_dtypes(include=["object", "string"]).columns
                  if pd.to_numeric(data[k].astype(str).str.replace(r"Rp|\s|\.", "", regex=True),
                                   errors="coerce").notna().mean() > 0.8]
    if teks_angka:
        print(f"Kolom teks yang isinya angka: {teks_angka}")
        print("  -> periksa: angka sungguhan yang tersimpan sebagai teks, atau label seperti kode pos?")

# Coba pada dataset latihan
cek_kualitas(df)
profil_kolom(df)

**Langkah untuk dataset Kaggle-mu:**
1. Unduh CSV dari Kaggle, unggah ke Colab (ikon folder di panel kiri).
2. Ganti nama file di sel bawah, lalu jalankan.
3. Salin tabel profil ke dokumen tugas dan lengkapi tiga kolom "isi sendiri".

In [ ]:
NAMA_FILE = "nama_file_kaggle.csv"   # <- ganti

import os
if os.path.exists(NAMA_FILE):
    data_tugas = pd.read_csv(NAMA_FILE)
    cek_kualitas(data_tugas)
    display(profil_kolom(data_tugas))
    # profil_kolom(data_tugas).to_excel("profil_tugas1.xlsx", index=False)   # simpan untuk dilengkapi
else:
    print(f"File '{NAMA_FILE}' belum ada. Unggah dulu dataset Kaggle-mu.")

**Rubrik Tugas 1:** ketepatan klasifikasi 40% · masalah kualitas data 25% · identifikasi data pribadi 20% · pertanyaan bisnis 15%. Format PDF maks. 3 halaman + file dataset, dikumpulkan di LMS folder Tugas 1.

---
## Rangkuman
- Statistika mengubah jejak data pelanggan menjadi keputusan, dari deskriptif hingga preskriptif.
- Jenis dan skala data (NOIR) menentukan statistik dan grafik yang boleh dipakai. Tipe kolom di komputer belum tentu sama dengan jenis data secara statistik.
- Pivot table dan grafik batang adalah alat pertama untuk meringkas data kategorik.
- Kualitas data harus diperiksa sebelum dipakai: akurat, lengkap, konsisten, valid, tepat waktu.
- UU PDP mewajibkan dasar pemrosesan yang sah, keamanan, dan notifikasi 3×24 jam bila bocor. Analis menerapkan minimisasi dan pseudonimisasi.

**Pertemuan 2 · Data wrangling & EDA:** bawa dataset Tugas 1 dan baca Anderson dkk. bab 1 (*Data and Statistics*).